In [8]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever
from dotenv import load_dotenv

load_dotenv()

True

### Documents

In [3]:
chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",
    
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

print(f"Created {len(chunks)} sample chunks for demonstration")

Created 60 sample chunks for demonstration


### Langchain Document => Chunks

In [4]:
# Convert to Document objects
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

### Hybrid Search

In [5]:
# Vector Retriever 

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 15})

# Keyword Retriever

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 15

# Hybrid Retriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

print("Setup complete!\n")

Setup complete!



In [6]:
# Query search 

query = "Tesla financial performance and production updates"

print("STEP 1: Hybrid Search Results")
print("-"*50)

retrieved_docs = hybrid_retriever.invoke(query)  # Get top 25 for reranking

# Show top 10 from hybrid search
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print(f"\n(Retrieved {len(retrieved_docs)} total chunks for reranking)\n")

STEP 1: Hybrid Search Results
--------------------------------------------------
 1. Tesla announced plans to expand Gigafactory production capacity.
 2. Tesla reported strong free cash flow generation of $7.5 billion.
 3. Tesla stock price reached new highs following earnings announcement.
 4. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 5. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 6. Tesla continues to lead in electric vehicle market share globally.
 7. Tesla Model Y became the best-selling vehicle worldwide.
 8. Financial markets showed volatility during earnings season.
 9. Tesla's automotive gross margin improved to 19.3% this quarter.
10. Tesla's Full Self-Driving revenue increased significantly.
11. Tesla's energy storage business grew 40% year-over-year.
12. Quarterly reports show mixed results across tech sector.
13. NVIDIA's automotive AI platform partnerships expanded.
14. Stock market analysts upgrade technology secto

### Reranker

In [ ]:
print("STEP 2: After Cohere Reranking (Top 10)")
print("-"*50)

# Initialize FlashRank re-ranker and get top 10 docs
compressor = FlashrankRerank(model="ms-marco-MultiBERT-L-12", top_n=10)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=hybrid_retriever
)

# Re-rank the retrieved documents
reranked_docs = compression_retriever.invoke(query)

# Show re-ranked results
for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...


STEP 2: After Cohere Reranking (Top 10)
--------------------------------------------------


ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:12<00:00, 8.37MiB/s]
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


 1. Tesla announced plans to expand Gigafactory production capacity.
 2. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 3. Tesla Model Y became the best-selling vehicle worldwide.
 4. Tesla continues to lead in electric vehicle market share globally.
 5. Tesla's energy storage business grew 40% year-over-year.
 6. Tesla's Full Self-Driving revenue increased significantly.
 7. Financial markets showed volatility during earnings season.
 8. Technology conferences showcase latest innovations.
 9. Tesla reported strong free cash flow generation of $7.5 billion.
10. Tesla's automotive gross margin improved to 19.3% this quarter.


### Final LLM Response

In [10]:
# Final LLM Response

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

print("\n" + "="*80)
print("FINAL: RAG with Reranked Context")
print("-"*40)

# Use top 5 reranked documents for final answer
top_reranked = reranked_docs[:5]

combined_input = f"""
Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in top_reranked])}

Please provide a clear, helpful answer using only the information from these documents.
"""

model = ChatOpenAI(model="gpt-4o")
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

result = model.invoke(messages)
print("Generated Response:")
print(result.content)


FINAL: RAG with Reranked Context
----------------------------------------


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generated Response:
Tesla has shown strong financial performance and production updates. The company announced plans to expand the production capacity of its Gigafactories, indicating a focus on increasing overall output. The production of the Tesla Cybertruck is set to ramp up in 2024 with initial deliveries beginning, reflecting ongoing developments in their vehicle lineup. Furthermore, the Tesla Model Y became the best-selling vehicle worldwide, demonstrating successful sales performance. Tesla continues to lead in global electric vehicle market share, highlighting its dominant position in the industry. Additionally, the company's energy storage business experienced significant growth, with a 40% increase year-over-year, showing diversification and strength in different market segments.
